# Field Representative Incentive Anomaly Detection

This notebook is a lightweight, reproducible review of the artifacts generated by `python run_pipeline.py`. The production implementation lives in `src/field_rep_anomaly`; the notebook does not duplicate model logic. An anomaly is a review signal, not evidence of fraud or incorrect payment.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
METRICS = ROOT / 'artifacts' / 'metrics'
REPORTS = ROOT / 'artifacts' / 'reports'
DATA = ROOT / 'data' / 'processed'
assert (METRICS / 'clustering_benchmark.csv').exists(), 'Run python run_pipeline.py first.'

## Provenance and analytical population

In [ ]:
metadata = json.loads((REPORTS / 'run_metadata.json').read_text(encoding='utf-8'))
scored = pd.read_csv(DATA / 'scored_observations.csv', parse_dates=['date'])
display(pd.Series(metadata, name='value').to_frame())
display(pd.DataFrame({'measure': ['rows', 'representatives', 'territories', 'products', 'injected anomaly rate'], 'value': [len(scored), scored.rep_id.nunique(), scored.territory_id.nunique(), scored.product_name.nunique(), scored.injected_anomaly_flag.mean()]}))

## Executed K-Means vs DBSCAN benchmark

In [ ]:
benchmark = pd.read_csv(METRICS / 'clustering_benchmark.csv')
selection = pd.read_csv(METRICS / 'model_selection.csv')
display(benchmark)
display(selection[['model', 'segmentation_score', 'anomaly_score', 'best_segmentation_model', 'best_anomaly_detection_model']])

## Cluster profiles and investigation queue

In [ ]:
profiles = pd.read_csv(METRICS / 'cluster_profiles.csv')
display(profiles[['model', 'cluster', 'population', 'population_pct', 'anomaly_rate', 'dominant_product', 'dominant_geography', 'business_interpretation']])
columns = ['rep_id', 'date', 'product_name', 'territory_id', 'total_sales', 'target_attainment_pct', 'actual_incentive_paid', 'anomaly_score', 'model', 'top_anomaly_drivers']
display(scored.nlargest(10, 'anomaly_score')[columns])

## Interpretation guardrails

The benchmark uses controlled synthetic anomalies for objective demo metrics. These labels were never model inputs. Synthetic-label performance may be optimistic and must be revalidated with governed review outcomes before operational use. DBSCAN is deterministic for fixed data and parameters; its reported stability measures sensitivity to small feature perturbations.